In [4]:
# Hierarchical Classification
import pandas as pd
import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModel, Trainer, TrainingArguments, DataCollatorWithPadding
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
import numpy as np
from safetensors.torch import save_file

In [6]:
# ---------- Statistics ----------
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("vinai/bertweet-base", use_fast=True)

# Load dataset
df = pd.read_csv("/kaggle/input/train-task1/full_train.csv")

# Tokenize and get token lengths
token_lengths = df['text'].apply(lambda x: len(tokenizer.tokenize(str(x))))

# Compute statistics
print("📊 Token Length Statistics:")
print(f"Min       : {token_lengths.min()}")
print(f"Max       : {token_lengths.max()}")
print(f"Mean      : {token_lengths.mean():.2f}")
print(f"85th pct. : {np.percentile(token_lengths, 85):.0f}")
print(f"88th pct. : {np.percentile(token_lengths, 88):.0f}")
print(f"90th pct. : {np.percentile(token_lengths, 90):.0f}")
print(f"95th pct. : {np.percentile(token_lengths, 95):.0f}")

📊 Token Length Statistics:
Min       : 0
Max       : 10658
Mean      : 58.32
85th pct. : 87
88th pct. : 110
90th pct. : 132
95th pct. : 201


In [8]:
# ---------- Configuration ----------
MODEL_NAME = "vinai/bertweet-base"
MAX_LEN = 128
BATCH_SIZE = 16
EPOCHS = 5

In [10]:
# ---------- Load Dataset ----------
df = pd.read_csv("/kaggle/input/train-task1/combined_train.csv")

train_df, val_df = train_test_split(df, test_size=0.15, stratify=df['Level 1'], random_state=42)

print(f"\nTraining set size: {len(train_df)}")
print(f"Validation set size: {len(val_df)}")


Training set size: 10929
Validation set size: 1929


In [11]:
# ---------- Dataset Class ----------
class CryptoQADataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len=128):
        self.df = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        text = str(self.df.loc[idx, 'text'])
        level1 = self.df.loc[idx, 'Level 1']
        level2 = self.df.loc[idx, 'Level 2']
        level3 = self.df.loc[idx, 'Level 3']

        encoded = self.tokenizer(text, truncation=True, padding='max_length', max_length=self.max_len)

        return {
            'input_ids': torch.tensor(encoded['input_ids']),
            'attention_mask': torch.tensor(encoded['attention_mask']),
            'level1': torch.tensor(level1, dtype=torch.long),
            'level2': torch.tensor(level2 if level1 == 2 else -1, dtype=torch.long),
            'level3': torch.tensor(level3 if (level1 == 2 and level2 == 0) else -1, dtype=torch.long),
        }

In [12]:
# ---------- Model ----------
class HierarchicalClassifier(nn.Module):
    def __init__(self, model_name="vinai/bertweet-base", hidden_size=768):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(0.3)
        self.classifier1 = nn.Linear(hidden_size, 3)
        self.classifier2 = nn.Linear(hidden_size, 3)
        self.classifier3 = nn.Linear(hidden_size, 4)

    def forward(self, input_ids, attention_mask, level1=None, level2=None, level3=None):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        pooled = self.dropout(outputs.last_hidden_state[:, 0])

        logits1 = self.classifier1(pooled)
        logits2 = self.classifier2(pooled)
        logits3 = self.classifier3(pooled)

        loss1 = loss2 = loss3 = 0.0
        loss_fct = nn.CrossEntropyLoss(reduction='none')

        if level1 is not None:
            loss1 = loss_fct(logits1, level1).mean()

            subjective_mask = (level1 == 2)
            if subjective_mask.sum() > 0 and level2 is not None:
                loss2 = loss_fct(logits2[subjective_mask], level2[subjective_mask]).mean()

                neutral_subjective_mask = (level2 == 0) & subjective_mask
                if neutral_subjective_mask.sum() > 0 and level3 is not None:
                    loss3 = loss_fct(logits3[neutral_subjective_mask], level3[neutral_subjective_mask]).mean()

        total_loss = loss1 + loss2 + loss3
        return {'loss': total_loss, 'logits': (logits1, logits2, logits3)}


In [13]:
# ---------- Data Collator ----------
def custom_collate(batch):
    keys = batch[0].keys()
    return {key: torch.stack([x[key] for x in batch]) for key in keys}

In [14]:
# ---------- Compute Metrics ----------
def compute_metrics(eval_preds):
    import numpy as np
    from sklearn.metrics import accuracy_score, f1_score

    logits1, logits2, logits3 = eval_preds.predictions
    level1_preds = np.argmax(logits1, axis=1)
    level2_preds = np.argmax(logits2, axis=1)
    level3_preds = np.argmax(logits3, axis=1)

    level1_labels, level2_labels, level3_labels = eval_preds.label_ids
    metrics = {}
    # Level 1 metrics
    acc1 = accuracy_score(level1_labels, level1_preds)
    f1_1 = f1_score(level1_labels, level1_preds, average='macro')
    metrics['level1_acc'] = acc1
    metrics['level1_f1'] = f1_1
    # Level 2 (only where Level 1 == 2)
    level2_mask = level1_labels == 2
    if np.sum(level2_mask) > 0:
        acc2 = accuracy_score(level2_labels[level2_mask], level2_preds[level2_mask])
        f1_2 = f1_score(level2_labels[level2_mask], level2_preds[level2_mask], average='macro')
        metrics['level2_acc'] = acc2
        metrics['level2_f1'] = f1_2
    # Level 3 (only where Level 1 == 2 and Level 2 == 0)
    level3_mask = (level1_labels == 2) & (level2_labels == 0)
    if np.sum(level3_mask) > 0:
        acc3 = accuracy_score(level3_labels[level3_mask], level3_preds[level3_mask])
        f1_3 = f1_score(level3_labels[level3_mask], level3_preds[level3_mask], average='macro')
        metrics['level3_acc'] = acc3
        metrics['level3_f1'] = f1_3
    metrics['comb'] = metrics['level1_acc']+metrics['level2_acc']+metrics['level3_acc']+metrics['level3_f1']+metrics['level2_f1']+metrics['level1_f1']
    return metrics

In [15]:
# ---------- Datasets & Trainer ----------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
train_dataset = CryptoQADataset(train_df, tokenizer)
val_dataset = CryptoQADataset(val_df, tokenizer)

model = HierarchicalClassifier()
training_args = TrainingArguments(
    output_dir="./hierarchical_output",
    eval_strategy="epoch",
    save_strategy="epoch",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=2e-5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="comb",
    report_to="none",
    label_names=["level1", "level2", "level3"],
    logging_dir="./logs"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=custom_collate,
    compute_metrics=compute_metrics,
)
print("Starting training")
trainer.train()

pytorch_model.bin:   0%|          | 0.00/543M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/543M [00:00<?, ?B/s]

Starting training


/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Epoch,Training Loss,Validation Loss,Level1 Acc,Level1 F1,Level2 Acc,Level2 F1,Level3 Acc,Level3 F1,Comb
1,No log,1.480703,0.804562,0.714908,0.811652,0.534703,0.839390,0.675165,4.380380
2,1.710500,1.334794,0.819077,0.745792,0.821229,0.569389,0.856975,0.737110,4.549573
3,1.147700,1.257883,0.829445,0.760951,0.836393,0.637044,0.867526,0.742322,4.673682
4,1.147700,1.301699,0.826335,0.759043,0.826018,0.666940,0.878077,0.779991,4.736404
5,0.882200,1.297444,0.824261,0.754899,0.831604,0.655655,0.869871,0.751677,4.687967


/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


TrainOutput(global_step=1710, training_loss=1.1902938842773438, metrics={'train_runtime': 905.681, 'train_samples_per_second': 60.336, 'train_steps_per_second': 1.888, 'total_flos': 0.0, 'train_loss': 1.1902938842773438, 'epoch': 5.0})

In [16]:
# ---------- Save Model ----------
import os 
os.makedirs("./safetensors", exist_ok=True)
save_file(model.state_dict(), "./safetensors/hierarchical_model.safetensors")

In [17]:
# ---------- Evaluate the final model on the test set ----------
print("\nEvaluating model on the test set...")
test_df = pd.read_csv("/kaggle/input/train-task1/reddit_test.csv")
test_df = test_df.rename(columns={'MAIN': 'text'})
test_dataset = CryptoQADataset(test_df, tokenizer)
test_results = trainer.evaluate(test_dataset)
print(f"Test set results: {test_results}")


Evaluating model on the test set...


/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Test set results: {'eval_loss': 1.0686464309692383, 'eval_level1_acc': 0.898, 'eval_level1_f1': 0.8179598649937633, 'eval_level2_acc': 0.8649350649350649, 'eval_level2_f1': 0.6262560323443329, 'eval_level3_acc': 0.8679245283018868, 'eval_level3_f1': 0.6460241856526686, 'eval_comb': 4.721099676227716, 'eval_runtime': 3.189, 'eval_samples_per_second': 156.788, 'eval_steps_per_second': 5.017, 'epoch': 5.0}


In [18]:
# ---------- Evaluate the final model on the test set ----------
print("\nEvaluating model on the test set...")
test_df = pd.read_csv("/kaggle/input/train-task1/twitter_test.csv")
test_df = test_df.rename(columns={'Tweet': 'text'})
test_dataset = CryptoQADataset(test_df, tokenizer)
test_results = trainer.evaluate(test_dataset)
print(f"Test set results: {test_results}")


Evaluating model on the test set...


/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Test set results: {'eval_loss': 2.0277464389801025, 'eval_level1_acc': 0.682983682983683, 'eval_level1_f1': 0.6800075102504769, 'eval_level2_acc': 0.736, 'eval_level2_f1': 0.5598689598689599, 'eval_level3_acc': 0.7555555555555555, 'eval_level3_f1': 0.653432490932491, 'eval_comb': 4.067848199591166, 'eval_runtime': 2.3798, 'eval_samples_per_second': 180.269, 'eval_steps_per_second': 5.883, 'epoch': 5.0}


In [19]:
# ---------- Evaluate the final model on the test set ----------
print("\nEvaluating model on the test set...")
test_df = pd.read_csv("/kaggle/input/train-task1/youtube_test.csv")
test_df = test_df.rename(columns={'comment': 'text'})
test_dataset = CryptoQADataset(test_df, tokenizer)
test_results = trainer.evaluate(test_dataset)
print(f"Test set results: {test_results}")


Evaluating model on the test set...


/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Test set results: {'eval_loss': 1.0289039611816406, 'eval_level1_acc': 0.882, 'eval_level1_f1': 0.580493430241554, 'eval_level2_acc': 0.8492822966507177, 'eval_level2_f1': 0.6985895894920358, 'eval_level3_acc': 0.925, 'eval_level3_f1': 0.4662819851043434, 'eval_comb': 4.401647301488651, 'eval_runtime': 2.7202, 'eval_samples_per_second': 183.811, 'eval_steps_per_second': 5.882, 'epoch': 5.0}


In [ ]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset

class CryptoQADataset(Dataset):
    def __init__(self, encodings, labels=None):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        # This part for labels is skipped during inference as labels=None
        if self.labels:
            item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.encodings.input_ids)

def run_inference_and_save(file_path, original_input_text_col, output_filename, output_columns_spec):
    
    print(f"\n--- Processing {file_path} ---")
    try:
        df = pd.read_csv(file_path)
        print(f"Original columns in {file_path}: {df.columns.tolist()}")
    except FileNotFoundError:
        print(f"Error: File not found at {file_path}. Please check the path and try again.")
        return

    # --- Prepare the 'text' column for the model input ---
    # Rename the specified input column (e.g., 'MAIN', 'Tweet', 'comment') to 'text' for consistency
    if original_input_text_col in df.columns:
        df = df.rename(columns={original_input_text_col: 'text'})
        print(f"Renamed '{original_input_text_col}' to 'text' for model input.")
    else:
        print(f"Error: Input text column '{original_input_text_col}' not found in {file_path}. Skipping inference.")
        return

    # Tokenize the data
    encodings = tokenizer(df['text'].tolist(), truncation=True, padding='max_length', max_length=512, return_tensors="pt")
    
    # Create the dataset for inference using your CryptoQADataset
    inference_dataset = CryptoQADataset(encodings)

    print("Running inference...")
    prediction_output = trainer.predict(inference_dataset)
    num_samples = len(df)
    


    # Assuming prediction_output.predictions is a 2D numpy array of logits (num_samples, total_num_classes)
    # where total_num_classes = num_L1_classes + num_L2_classes + num_L3_classes
    
    # Slice the predictions (ADJUST INDICES BASED ON YOUR MODEL'S ACTUAL OUTPUT STRUCTURE)
    # For instance, if L1 has 5 classes (0-4), L2 has 3 (5-7), L3 has 3 (8-10)
    # total_classes = 5 + 3 + 3 = 11
    # raw_predictions = prediction_output.predictions # Your model's actual raw predictions
    # level1_logits = raw_predictions[:, 0:5]   # First 5 columns for Level 1
    # level2_logits = raw_predictions[:, 5:8]   # Next 3 columns for Level 2
    # level3_logits = raw_predictions[:, 8:11] # Next 3 columns for Level 3

    # # Get predicted IDs by argmax
    # predicted_level1_ids = np.argmax(level1_logits, axis=1)
    # predicted_level2_ids = np.argmax(level2_logits, axis=1)
    # predicted_level3_ids = np.argmax(level3_logits, axis=1)

    # # Map IDs to string labels
    # predicted_level1 = [level1_id_to_label[idx] for idx in predicted_level1_ids]
    # predicted_level2 = [level2_id_to_label[idx] for idx in predicted_level2_ids]
    # predicted_level3 = [level3_id_to_label[idx] for idx in predicted_level3_ids]

    # As a temporary placeholder, generating random choices:
    predicted_level1 = np.random.choice(list(level1_id_to_label.values()), size=num_samples)
    predicted_level2 = np.random.choice(list(level2_id_to_label.values()), size=num_samples)
    predicted_level3 = np.random.choice(list(level3_id_to_label.values()), size=num_samples)

    # --- END: YOU MUST REPLACE THIS SECTION ---

    # Create the output DataFrame
    output_df = pd.DataFrame()

    # Add original columns as specified in output_columns_spec
    # This loop ensures that all original columns (like title, selftext, comment_id, MAIN, Text)
    # are carried over if they exist in the input dataframe.
    for col_name in output_columns_spec:
        # Special handling for 'Text' for Twitter output, which was 'Tweet' input
        if col_name == 'Text' and original_input_text_col == 'Tweet' and 'text' in df.columns:
            output_df['Text'] = df['text'] # Use the 'text' column content
        elif col_name == 'MAIN' and original_input_text_col == 'MAIN' and 'text' in df.columns:
            output_df['MAIN'] = df['text'] # Carry over the 'MAIN' input column
        elif col_name in df.columns:
            output_df[col_name] = df[col_name]
        # Predicted columns will be added below

    # Add predicted columns
    # 'MAIN' is an input column, not a predicted one from this model, so no 'predicted_main'
    
    # Handle Level 1 vs Level1 based on output filename for consistency in requested output
    if output_filename == "crypto_test_youtube.csv":
        output_df['Level1'] = predicted_level1
    else: # For Reddit and Twitter, use 'Level 1'
        output_df['Level 1'] = predicted_level1 
        
    output_df['Level 2'] = predicted_level2
    output_df['Level 3'] = predicted_level3
    
    # Select and reorder columns for the final output DataFrame based on output_columns_spec
    final_output_df = pd.DataFrame()
    for col_name in output_columns_spec:
        if col_name in output_df.columns:
            final_output_df[col_name] = output_df[col_name]
        else:
            print(f"Warning: Column '{col_name}' requested for output but not found in prepared DataFrame for '{output_filename}'. It will be missing.")

    # Save the results
    final_output_df.to_csv(output_filename, index=False)
    print(f"Inference complete and results saved to '{output_filename}'")
    print(f"Columns in '{output_filename}': {final_output_df.columns.tolist()}")
    print(f"First 5 rows of '{output_filename}':\n{final_output_df.head()}")


# --- Inference Calls for Each Dataset ---

# 1. Reddit Dataset
# Input is 'MAIN', outputs are Level 1, 2, 3. Keep original 'title' and 'selftext'.
run_inference_and_save(
    file_path="/kaggle/input/train-task1/CRYPTO_REDDIT_TEST.csv",
    original_input_text_col='MAIN', # Corrected: Input column for model is 'MAIN'
    output_filename="crypto_test_reddit.csv",
    output_columns_spec=['title', 'selftext', 'MAIN', 'level 1', 'level 2', 'level 3']
)

# 2. Twitter Dataset
# Input is 'Tweet', outputs are Level 1, 2, 3. Output text as 'Text'.
run_inference_and_save(
    file_path="/kaggle/input/train-task1/CRYPTO_TWITTER_TEST.csv",
    original_input_text_col='Tweet', # Input column for model is 'Tweet'
    output_filename="crypto_test_tweet.csv",
    output_columns_spec=['Text', 'Level 1', 'Level 2', 'Level 3']
)

# 3. YouTube Dataset
# Input is 'comment', outputs are Level 1, 2, 3. Keep original 'comment_id'.
run_inference_and_save(
    file_path="/kaggle/input/train-task1/CRYPTO_YOUTUBE_TEST.csv",
    original_input_text_col='comment', # Input column for model is 'comment'
    output_filename="crypto_test_youtube.csv",
    output_columns_spec=['comment_id', 'MAIN', 'Level1', 'Level 2', 'Level 3']
)

print("\nAll inference tasks completed (assuming files were found and model components are defined).")